# GenAI search with DuckDuckGo and Tavily

This notebook runs an LLM (via [OpenRouter](https://openrouter.ai)) with a `search` tool. The model decides when to call the tool; the tool itself searches DuckDuckGo first and falls back to Tavily if DuckDuckGo fails.

Requires a `.env` file (copy `.env.example`) with `OPENROUTER_API_KEY` and, optionally, `TAVILY_API_KEY` for the fallback.

In [ ]:
import os
from pprint import pprint

from ddgs.exceptions import DDGSException
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

load_dotenv()


### `search_duckduckgo`

Uses LangChain's `DuckDuckGoSearchResults` tool (from `langchain-community`) to run a DuckDuckGo search and return the raw list of result dicts. This is the primary (free, no API key) search provider.

In [ ]:
def search_duckduckgo(query: str, max_results: int = 5) -> list[dict]:
    api_wrapper = DuckDuckGoSearchAPIWrapper(max_results=max_results)
    return DuckDuckGoSearchResults(api_wrapper=api_wrapper, output_format="list").invoke(query)

### `search_tavily`

Uses LangChain's `TavilySearch` tool (from `langchain-tavily`) to run a search via the Tavily API. Requires the `TAVILY_API_KEY` environment variable to be set; raises a clear error if it isn't. Used as the fallback provider when DuckDuckGo fails.

In [ ]:
def search_tavily(query: str, max_results: int = 5) -> list[dict]:
    if not os.environ.get("TAVILY_API_KEY"):
        raise RuntimeError("Set TAVILY_API_KEY to use Tavily search.")

    response = TavilySearch(max_results=max_results).invoke(query)
    return response.get("results", [])

### `search`

The tool the LLM calls. Tries DuckDuckGo first, and falls back to Tavily if DuckDuckGo raises a `DDGSException`. Returns a dict noting which provider actually served the results. The `@tool` decorator turns the docstring below into the description the model sees.

In [ ]:
@tool
def search(query: str, max_results: int = 5) -> dict:
    """Search the web for current information. Uses DuckDuckGo, falling back to Tavily if DuckDuckGo fails."""
    try:
        return {
            "provider": "duckduckgo",
            "results": search_duckduckgo(query=query, max_results=max_results),
        }
    except DDGSException as duckduckgo_error:
        return {
            "provider": "tavily",
            "results": search_tavily(query=query, max_results=max_results),
            "duckduckgo_error": str(duckduckgo_error),
        }

### OpenRouter client

Connects to OpenRouter through LangChain's `ChatOpenAI`, pointed at OpenRouter's OpenAI-compatible endpoint. Reads `OPENROUTER_API_KEY` from the environment (loaded from `.env` above).

In [ ]:
openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise RuntimeError("Set OPENROUTER_API_KEY (see .env.example) to run inference.")

MODEL = "anthropic/claude-haiku-4.5"
llm = ChatOpenAI(
    model=MODEL,
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
    max_tokens=1024,
)

### Bind the tool

Registers `search` as the tool available to the model.

In [ ]:
tools = [search]
# tools = [] # Uncomment to run inference without any tools - no web search
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

### `run_agent`

The inference loop: sends the conversation to the model via `llm_with_tools`, and whenever it responds with a `search` tool call, invokes the matching LangChain tool and feeds the result back in as a `ToolMessage`. Repeats until the model returns a plain text answer (or `max_turns` is hit).

In [ ]:
def run_agent(user_prompt: str, max_turns: int = 5) -> str:
    messages = [HumanMessage(user_prompt)]

    for _ in range(max_turns):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:
            result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    return "Max turns reached without a final answer."

In [ ]:
answer = run_agent("What's the latest model put out by OpenAI?")

In [ ]:
pprint(answer)